# Exercise 4: Generation

*Finite-State Morphology · version 0.1.0*

> With the Apertium Turkish generator, generate the six case forms of *kitap* in singular and plural, and explain the final-consonant alternation. Then generate the case forms of the place name *Ankara* (tag `<np><top>`): how does vowel-final attachment work, and does the generator write the apostrophe that Turkish spelling requires after proper names (*Ankara'da*)? Finally, use `substitute_entity()` (adapted to Turkish) on three sentences, and check every generated sentence by hand.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. These notebooks compile lexc and twolc grammars, which needs the compiled `hfst` package; on PyPI it had wheels for Python 3.11 but not for 3.13 when checked on 19 October 2026, so use Python 3.11 or 3.12 if `pip` cannot find it. The Apertium transducers are downloaded once with `turkicnlp.download`.

In [ ]:
import tempfile
from pathlib import Path
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

def compile_grammar(lexc_text, twolc_text=None):
    """Compile a lexc lexicon (and optional twolc rules); return (analyser, generator) in lookup format."""
    work = Path(tempfile.mkdtemp())
    (work / "g.lexc").write_text(lexc_text, encoding="utf-8")
    gen = hfst.compile_lexc_file(str(work / "g.lexc"))
    if twolc_text:
        (work / "g.twolc").write_text(twolc_text, encoding="utf-8")
        hfst.compile_twolc_file(str(work / "g.twolc"), str(work / "g.twol.hfst"))
        stream, rules = hfst.HfstInputStream(str(work / "g.twol.hfst")), []
        while not stream.is_eof():
            rules.append(stream.read())
        stream.close()
        gen.compose_intersect(tuple(rules))
    gen.minimize()
    ana = hfst.HfstTransducer(gen)
    ana.invert()
    ana.minimize()
    for t in (ana, gen):
        t.convert(hfst.ImplementationType.HFST_OLW_TYPE)
    return ana, gen

def apertium(lang, script, kind="automorf"):
    """An Apertium transducer distributed with TurkicNLP (downloaded on first use)."""
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.{kind}.hfst"
    stream = hfst.HfstInputStream(str(path))
    fst = stream.read()
    stream.close()
    return fst

lookup = lambda fst, s: sorted({out for out, _ in fst.lookup(s)})

In [ ]:
gen = apertium("tur", "Latn", "autogen")
ana = apertium("tur", "Latn")
CASES = ["nom", "acc", "dat", "loc", "abl", "gen"]
for number in ["", "<pl>"]:
    print("kitap" + (number or " (sg)"), [lookup(gen, f"kitap<n>{number}<{c}>") for c in CASES])
print("Ankara", [lookup(gen, f"Ankara<np><top><{c}>") for c in CASES])
print("Ankarada analysed:", lookup(ana, "Ankarada"), "| Ankara'da:", lookup(ana, "Ankara'da"))

`substitute_entity()` from the chapter needs no change for Turkish other than passing the Turkish transducers. The three sentences each contain one location in a different case.

In [ ]:
def substitute_entity(tokens, labels, idx, new_lemma, allowed_lemmas, ana, gen):
    """The chapter's conservative single-token B-LOC replacement; None means abstain."""
    if len(tokens) != len(labels) or not 0 <= idx < len(tokens):
        raise ValueError("unaligned tokens/labels or invalid index")
    if labels[idx] != "B-LOC" or (idx + 1 < len(labels) and labels[idx + 1] == "I-LOC"):
        return None
    if new_lemma not in allowed_lemmas or any(c.isspace() for c in new_lemma):
        return None
    readings = {a for a, _ in ana.lookup(tokens[idx]) if "<np><top>" in a and "+" not in a}
    if len(readings) != 1:
        return None
    _, tags = readings.pop().split("<", 1)
    requested = f"{new_lemma}<{tags}"
    forms = {f for f, _ in gen.lookup(requested)}
    if len(forms) != 1:
        return None
    form = forms.pop()
    if not form or any(c.isspace() for c in form) or requested not in {a for a, _ in ana.lookup(form)}:
        return None
    new_tokens = list(tokens)
    new_tokens[idx] = form
    return new_tokens, list(labels)

cities = {"İzmir", "Bursa", "Konya"}          # demonstration list, not a vetted gazetteer
SENTENCES = [(["Ali", "Ankara'dan", "geldi", "."], 1),
             (["Toplantı", "İstanbul'da", "yapıldı", "."], 1),
             (["Yarın", "Bursa'ya", "gidiyorum", "."], 1)]
for tokens, idx in SENTENCES:
    labels = ["O"] * len(tokens)
    labels[idx] = "B-LOC"
    for city in ["İzmir", "Bursa", "Konya", "Paris"]:
        out = substitute_entity(tokens, labels, idx, city, cities, ana, gen)
        print(" ".join(tokens), "->", city, ":", " ".join(out[0]) if out else None)

## Worked solution

Output of `experiments/ch06_exercises/run_ch06_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
kitap (sg) [['kitap'], ['kitabı'], ['kitaba'], ['kitapta'], ['kitaptan'], ['kitabın']]
kitap<pl> [['kitaplar'], ['kitapları'], ['kitaplara'], ['kitaplarda'], ['kitaplardan'], ['kitapların']]
Ankara [['Ankara'], ["Ankara'yı"], ["Ankara'ya"], ["Ankara'da"], ["Ankara'dan"], ["Ankara'nın"]]
Ankarada analysed: [] | Ankara'da: ['Ankara<np><top><loc>', 'Ankara<np><top><loc>+i<cop><aor><p3><sg>']
```

```
Ali Ankara'dan geldi . -> İzmir : Ali İzmir'den geldi .
Ali Ankara'dan geldi . -> Bursa : Ali Bursa'dan geldi .
Ali Ankara'dan geldi . -> Konya : Ali Konya'dan geldi .
Ali Ankara'dan geldi . -> Paris : None
Toplantı İstanbul'da yapıldı . -> İzmir : Toplantı İzmir'de yapıldı .
Toplantı İstanbul'da yapıldı . -> Bursa : Toplantı Bursa'da yapıldı .
Toplantı İstanbul'da yapıldı . -> Konya : Toplantı Konya'da yapıldı .
Toplantı İstanbul'da yapıldı . -> Paris : None
Yarın Bursa'ya gidiyorum . -> İzmir : Yarın İzmir'e gidiyorum .
Yarın Bursa'ya gidiyorum . -> Bursa : Yarın Bursa'ya gidiyorum .
Yarın Bursa'ya gidiyorum . -> Konya : Yarın Konya'ya gidiyorum .
Yarın Bursa'ya gidiyorum . -> Paris : None
```

**kitap.** The stem ends in *p* before a consonant-initial suffix or no suffix (*kitap*, *kitapta*, *kitaptan*, *kitaplar* …) and in *b* before a vowel-initial one (*kitabı*, *kitaba*, *kitabın*). The plural *-lar* begins with a consonant, so every plural form keeps *p*.

**Ankara.** As a vowel-final stem, *Ankara* takes the buffer *y* before the accusative and dative (*Ankara'yı*, *Ankara'ya*) and *n* before the genitive (*Ankara'nın*); the locative and ablative have *d* after the vowel and *a* after the back vowel (*Ankara'da*, *Ankara'dan*). The generator writes the apostrophe, and the analyser rejects *Ankarada* without it.

**Substitution.** Nine of the twelve calls return a sentence; *Paris* is rejected because it is not in the list. Checked by hand, all nine are correct Turkish: *İzmir'den*, *Bursa'dan*, *Konya'dan*; *İzmir'de*, *Bursa'da*, *Konya'da*; *İzmir'e* (no buffer after a consonant), *Bursa'ya*, *Konya'ya*. One of them replaces *Bursa* with *Bursa*, which gives back the original sentence; an augmentation script should skip the original lemma.

## Your turn — extension

1. Add *Çorum* and *Kars* to the list and to a sentence with a locative. Check *Çorum'da* and *Kars'ta* by hand.
2. What happens when the original entity is a two-word name (*Abu Dabi*), which the function rejects because the next label is `I-LOC`? Write the extension and its tests.